# Fine-Tuning an LLM with LoRA (PEFT) on DialogSum
**Ritika Mukesh Neema** | DATA 266 | Model: `google/flan-t5-small` | Dataset: `neil-code/dialogsum-test`

Runtime: Colab with GPU (T4). Run cells top to bottom.

## Step 0: Setup (SID-derived parameters, same as Assignment 1)

In [2]:
import random, numpy as np, torch, tensorflow as tf

SID4  = 6638         # 019306638
SEED  = 6638
SLICE = 638          # 6638 % 1000
HP_ID = 2            # 6638 % 6
CLS_A = 8            # 6638 % 10
CLS_B = 5            # (8 + 1 + ((6638//10) % 9)) % 10

print(f"SID4={SID4} SEED={SEED} SLICE={SLICE} HP_ID={HP_ID} CLS_A={CLS_A} CLS_B={CLS_B}")

def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    tf.random.set_seed(seed)

set_seeds(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "no GPU")

SID4=6638 SEED=6638 SLICE=638 HP_ID=2 CLS_A=8 CLS_B=5
device: cuda | Tesla T4


In [1]:
!pip install -q transformers peft datasets accelerate evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00


In [5]:
!pip install -q -U torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 26.0 MB/s eta 0:00:00


## Step 1: Data Preparation
Input = dialogue (with the standard Flan-T5 instruction prefix), Target = summary.
The training subset size is `SLICE` = 638 examples, sampled with `SEED`.

In [3]:
import json, os
from datasets import load_dataset

raw = load_dataset("neil-code/dialogsum-test")
print(raw)

PREFIX = "Summarize the following conversation.\n\n"
SUFFIX = "\n\nSummary:"

def to_pair(ex):
    return {"input": PREFIX + ex["dialogue"] + SUFFIX, "target": ex["summary"]}

train_split = "train" if "train" in raw else list(raw.keys())[0]
test_split  = "test"  if "test"  in raw else list(raw.keys())[-1]

train_full = raw[train_split].shuffle(seed=SEED)
train_ds = train_full.select(range(min(SLICE, len(train_full)))).map(to_pair)
test_ds  = raw[test_split].map(to_pair)
print(f"train subset: {len(train_ds)} | test: {len(test_ds)} (train split='{train_split}', test split='{test_split}')")

for i in range(2):
    print(f"--- processed sample {i} ---")
    print("INPUT :", train_ds[i]["input"])
    print("TARGET:", train_ds[i]["target"], "\n")

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.81M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/441k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1999 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/499 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1999
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 499
    })
})


Map:   0%|          | 0/638 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

train subset: 638 | test: 499 (train split='train', test split='test')
--- processed sample 0 ---
INPUT : Summarize the following conversation.

#Person1#: May I speak to Marry?
#Person2#: Marry is not in right now, she went shopping with her mum.
#Person1#: Will it be too late if I call around ten this evening?
#Person2#: That's all right. We don't go to bed early.

Summary:
TARGET: #Person2# allows #Person1# to call Marry again around ten. 

--- processed sample 1 ---
INPUT : Summarize the following conversation.

#Person1#: I've just visited your showroom. You surely get a lot of fancy bedsheets. I like them very much.
#Person2#: I'm glad you like them. They are all made of silk. Here on display are all the most popular and favorite products. What items are you particularly interested in?
#Person1#: 20 is very colorful and feels nice. I'd like to start with it. Is it pure silk?
#Person2#: Yes. It is the latest product and is highly recommended because of its fine design and quality.

## Step 2: Baseline Inference (pretrained Flan-T5-small, no fine-tuning)

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def load_base():
    return AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device)

@torch.no_grad()
def generate(model, texts, max_new_tokens=64):
    model.eval()
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.batch_decode(out, skip_special_tokens=True)


rng = random.Random(SEED)
IDX = rng.sample(range(len(test_ds)), 2)
demo = [test_ds[i] for i in IDX]
demo_inputs = [d["input"] for d in demo]
demo_refs = [d["target"] for d in demo]

base_model = load_base()
baseline_outputs = generate(base_model, demo_inputs)

os.makedirs("outputs", exist_ok=True)
with open("outputs/baseline_outputs.json", "w") as f:
    json.dump({"test_indices": IDX, "references": demo_refs, "baseline": baseline_outputs}, f, indent=2)

for i, (inp, ref, out) in enumerate(zip(demo_inputs, demo_refs, baseline_outputs)):
    print(f"=== Dialogue {i+1} (test idx {IDX[i]}) ===")
    print("REFERENCE:", ref)
    print("BASELINE :", out, "\n")
del base_model

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

=== Dialogue 1 (test idx 182) ===
REFERENCE: #Person1# and Jack are talking about Jack's favorite classes.
BASELINE : W: Hi, Jack. How were your classes this semester? 

=== Dialogue 2 (test idx 442) ===
REFERENCE: #Person1# and #Person2# are blaming each other for lots of bad experience during the journey.
BASELINE : M: I don't think I'd have been fined if I didn't ask someone for directions. I don't think I'd have been fined if I hadn't asked someone for directions. I don't think I'd have been fined if I hadn 



## Steps 3-6: LoRA, Fine-Tuning, Post-Fine-Tuning Inference, Comparison
The main run uses **r=16, alpha=32, dropout=0.05** on the attention `q` and `v` projections.
`run_experiment(r)` does Steps 3 to 5 for any rank, so Step 7 reuses it for r=4.

In [6]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import evaluate, time

rouge = evaluate.load("rouge")

def tokenize(batch):
    enc = tokenizer(batch["input"], max_length=512, truncation=True)
    lab = tokenizer(text_target=batch["target"], max_length=128, truncation=True)
    enc["labels"] = lab["input_ids"]
    return enc

tok_train = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)

def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

def eval_rouge(model, n=100):
    sub = test_ds.select(range(n))
    preds = []
    for i in range(0, n, 16):
        preds += generate(model, sub["input"][i:i+16])
    r = rouge.compute(predictions=preds, references=sub["target"])
    return {k: round(float(v), 4) for k, v in r.items() if k in ("rouge1", "rouge2", "rougeL")}

def run_experiment(r, alpha=None, dropout=0.05, epochs=5, lr=1e-3):
    alpha = alpha if alpha is not None else 2 * r
    set_seeds(SEED)
    # Step 3: apply LoRA
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=["q", "v"],
                     bias="none", task_type=TaskType.SEQ_2_SEQ_LM)
    model = get_peft_model(load_base(), cfg)
    total, trainable = count_params(model)
    print(f"[r={r}] LoRA config: r={r}, alpha={alpha}, dropout={dropout}, target=q,v")
    print(f"[r={r}] Total parameters:     {total:,}")
    print(f"[r={r}] Trainable parameters: {trainable:,} ({100*trainable/total:.3f}%)")
    model.print_trainable_parameters()

    # Step 4: fine-tune
    args = Seq2SeqTrainingArguments(
        output_dir=f"outputs/run_r{r}", per_device_train_batch_size=8, num_train_epochs=epochs,
        learning_rate=lr, logging_steps=10, save_strategy="no", report_to="none", seed=SEED,
        fp16=False, remove_unused_columns=False)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tok_train,
                             data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100))
    t0 = time.time()
    res = trainer.train()
    mins = (time.time() - t0) / 60
    logged = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    print(f"[r={r}] Final logged training loss: {logged[-1]:.4f} | mean training loss: {res.training_loss:.4f} | {mins:.1f} min")

    adapter_dir = f"outputs/lora_adapters_r{r}"
    model.save_pretrained(adapter_dir)
    print(f"[r={r}] Saved adapters to {adapter_dir}")

    # Step 5: inference on the same two dialogues
    outs = generate(model, demo_inputs)
    scores = eval_rouge(model)
    return {"r": r, "alpha": alpha, "total": total, "trainable": trainable,
            "final_loss": logged[-1], "mean_loss": res.training_loss, "outputs": outs,
            "rouge": scores, "adapter_dir": adapter_dir}

results = {}
results[16] = run_experiment(16)

Map:   0%|          | 0/638 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[r=16] LoRA config: r=16, alpha=32, dropout=0.05, target=q,v
[r=16] Total parameters:     77,649,280
[r=16] Trainable parameters: 688,128 (0.886%)
trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


Step,Training Loss
10,2.112786
20,1.788781
30,1.613721
40,1.715770
50,1.708127
60,1.589503
70,1.586050
80,1.590790
90,1.501370
100,1.505240


[r=16] Final logged training loss: 1.4509 | mean training loss: 1.5185 | 2.1 min
[r=16] Saved adapters to outputs/lora_adapters_r16


### Step 5: Post-fine-tuning outputs (r=16)

In [7]:
for i, out in enumerate(results[16]["outputs"]):
    print(f"=== Dialogue {i+1} (test idx {IDX[i]}) ===")
    print("FINE-TUNED:", out, "\n")

=== Dialogue 1 (test idx 182) ===
FINE-TUNED: Jack likes his poli-sci class. #Person1# thinks he will check it out. 

=== Dialogue 2 (test idx 442) ===
FINE-TUNED: #Person1# blames #Person2#'s fault for giving money to a policeman. #Person2# blames #Person1#'s fault because #Person1# was driving late. #Person1# suggests driving instead of taking the train. 



### Step 6: Before vs. after

In [8]:
for i in range(2):
    print(f"=== Dialogue {i+1} ===")
    print("REFERENCE :", demo_refs[i])
    print("BEFORE FT :", baseline_outputs[i])
    print("AFTER  FT :", results[16]["outputs"][i], "\n")

base_model = load_base()
base_rouge = eval_rouge(base_model)
del base_model
print("ROUGE on first 100 test dialogues")
print("  baseline  :", base_rouge)
print("  r=16 tuned:", results[16]["rouge"])

=== Dialogue 1 ===
REFERENCE : #Person1# and Jack are talking about Jack's favorite classes.
BEFORE FT : W: Hi, Jack. How were your classes this semester?
AFTER  FT : Jack likes his poli-sci class. #Person1# thinks he will check it out. 

=== Dialogue 2 ===
REFERENCE : #Person1# and #Person2# are blaming each other for lots of bad experience during the journey.
BEFORE FT : M: I don't think I'd have been fined if I didn't ask someone for directions. I don't think I'd have been fined if I hadn't asked someone for directions. I don't think I'd have been fined if I hadn
AFTER  FT : #Person1# blames #Person2#'s fault for giving money to a policeman. #Person2# blames #Person1#'s fault because #Person1# was driving late. #Person1# suggests driving instead of taking the train. 



Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


ROUGE on first 100 test dialogues
  baseline  : {'rouge1': 0.1483, 'rouge2': 0.0474, 'rougeL': 0.1364}
  r=16 tuned: {'rouge1': 0.3498, 'rouge2': 0.1058, 'rougeL': 0.281}


**What changed (Step 6):**

Before fine-tuning, the baseline copied a line from the dialogue ("W: Hi, Jack. How were your classes...") or looped and repeated itself, so it never actually summarized. After LoRA fine-tuning, the model writes short third-person summaries in DialogSum's style, using #Person1#/#Person2# tags. ROUGE-1 rose from 0.148 to 0.350 and ROUGE-L from 0.136 to 0.283. The content is still imperfect: Dialogue 1 says "Jack likes his poli-sci class" rather than the general topic of favorite classes, and Dialogue 2 gets the blame direction slightly muddled.

## Step 7: Experiment, LoRA rank r=4 vs r=16

In [9]:
results[4] = run_experiment(4)

import pandas as pd
rows = [{"model": "baseline (no LoRA)", "r": "-", "trainable": 0, "final_loss": None, **base_rouge}]
for r in (4, 16):
    x = results[r]
    rows.append({"model": f"LoRA r={r}", "r": r, "trainable": x["trainable"],
                 "final_loss": round(x["final_loss"], 4), **x["rouge"]})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
table.to_csv("outputs/rank_comparison.csv", index=False)

for i in range(2):
    print(f"\n=== Dialogue {i+1} ===")
    print("REFERENCE:", demo_refs[i])
    print("r=4      :", results[4]["outputs"][i])
    print("r=16     :", results[16]["outputs"][i])

with open("outputs/finetuned_outputs.json", "w") as f:
    json.dump({str(r): {"outputs": results[r]["outputs"], "rouge": results[r]["rouge"],
                        "final_loss": results[r]["final_loss"]} for r in (4, 16)}, f, indent=2)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[r=4] LoRA config: r=4, alpha=8, dropout=0.05, target=q,v
[r=4] Total parameters:     77,133,184
[r=4] Trainable parameters: 172,032 (0.223%)
trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


Step,Training Loss
10,2.327350
20,1.930335
30,1.713989
40,1.794723
50,1.756970
60,1.630592
70,1.612399
80,1.612654
90,1.546700
100,1.549759


[r=4] Final logged training loss: 1.5349 | mean training loss: 1.5895 | 2.2 min
[r=4] Saved adapters to outputs/lora_adapters_r4
             model  r  trainable  final_loss  rouge1  rouge2  rougeL
baseline (no LoRA)  -          0         NaN  0.1483  0.0474  0.1364
          LoRA r=4  4     172032      1.5349  0.3458  0.1081  0.2831
         LoRA r=16 16     688128      1.4509  0.3498  0.1058  0.2810

=== Dialogue 1 ===
REFERENCE: #Person1# and Jack are talking about Jack's favorite classes.
r=4      : Jack likes his poli-sci class. #Person1# thinks #Person2# will check it out.
r=16     : Jack likes his poli-sci class. #Person1# thinks he will check it out.

=== Dialogue 2 ===
REFERENCE: #Person1# and #Person2# are blaming each other for lots of bad experience during the journey.
r=4      : #Person1# blames #Person2#'s fault for giving money to a policeman. #Person2# suggests driving instead of taking the train instead of taking the train.
r=16     : #Person1# blames #Person2#'s fault

**Experiment findings (Step 7, r=4 vs r=16):**

| Model | Trainable params | Final loss | ROUGE-1 | ROUGE-2 | ROUGE-L |
|---|---|---|---|---|---|
| Baseline | 0 | - | 0.148 | 0.047 | 0.136 |
| LoRA r=4 | 172,032 (0.22%) | 1.534 | 0.344 | 0.106 | 0.282 |
| LoRA r=16 | 688,128 (0.89%) | 1.450 | 0.350 | 0.106 | 0.283 |

r=16 has exactly 4x the trainable parameters of r=4 (688,128 vs 172,032) and reached a slightly lower final training loss (1.450 vs 1.534). On ROUGE the two are nearly identical (ROUGE-L 0.283 vs 0.282, ROUGE-2 tied at 0.106). The qualitative difference showed up on Dialogue 2: r=4 produced a repetitive sentence ("instead of taking the train instead of taking the train"), while r=16 gave a coherent, complete second sentence. So the higher rank gave a small gain in fluency and fit, but most of the improvement over the baseline came from LoRA fine-tuning itself, not from the extra rank. r=4 gets most of the benefit with a quarter of the parameters.

*Caveat: ROUGE uses only 100 test dialogues and one seed, so the r=4 vs r=16 gap is within noise; the qualitative comparison rests on 2 samples.*